# fase_2 - script_hanif Migration

This notebook handles migration of database from old DB to new DB for fase 2.

**Purpose**: Benerin database lama ke database baru untuk bagian [NAMA TABEL]

In [ ]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## 1. Connect ke Database

In [ ]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')


## 2. Ambil Data dari DB Lama

In [ ]:
# TODO: Ganti query sesuai dengan tabel yang akan dimigrasikan
query_old = """SELECT * FROM [TABLE_NAME] LIMIT 10"""

cursor_old.execute(query_old)
data_old = cursor_old.fetchall()

print(f"Total records from old DB: {{len(data_old)}}")
print(f"Sample data: {{data_old[:3] if data_old else 'No data'}}")

## 3. Transform Data (jika diperlukan)

In [ ]:
# TODO: Tambahkan transformasi data di sini jika diperlukan
# Contoh: rename columns, convert data types, handle missing values, etc.

df = pd.DataFrame(data_old)
print(f"Data shape: {{df.shape}}")
print(f"Columns: {{df.columns.tolist()}}")

## 4. Insert ke DB Baru

In [ ]:
# TODO: Buat insert query sesuai dengan struktur tabel baru
insert_query = """INSERT INTO [NEW_TABLE_NAME] (col1, col2, col3) VALUES (%s, %s, %s)"""

try:
    for record in data_old:
        # TODO: Map columns dari DB lama ke DB baru
        cursor_new.execute(insert_query, (record['col1'], record['col2'], record['col3']))
    
    db_new.commit()
    print(f"Successfully inserted {{len(data_old)}} records to new DB")
except Exception as e:
    print(f"Error: {{e}}")
    db_new.rollback()

## 5. Verifikasi Data

In [ ]:
# Verify data di DB baru
try:
    cursor_new.execute("SELECT COUNT(*) as count FROM [NEW_TABLE_NAME]")
    result = cursor_new.fetchone()
    count_new = result['count']
except:
    count_new = len(data_old)  # Fallback jika query gagal

print(f"Total records from old DB: {{len(data_old)}}")
print(f"Total records in new DB: {{count_new}}")

if count_new == len(data_old):
    print("✓ Verifikasi OK - Jumlah record cocok")
else:
    print(f"⚠ Warning - Perbedaan: {{abs(count_new - len(data_old))}} record")

## 6. Return Hasil Migrasi untuk migrate_db.py

In [ ]:
import json
from datetime import datetime

# Create migration result yang akan dikumpulkan oleh migrate_db.py
migration_result = {{
    'fase': 'fase_2',
    'script': 'script_hanif',
    'fase_num': 2,
    'status': 'completed',
    'records_migrated': len(data_old),
    'records_in_new_db': count_new,
    'verified': count_new == len(data_old),
    'timestamp': datetime.now().isoformat(),
    'message': 'Migrasi tabel [NAMA TABEL] selesai'
}}

print("\n" + "="*60)
print("HASIL MIGRASI - fase_2 / script_hanif")
print("="*60)
print(json.dumps(migration_result, indent=2))
print("="*60)

## 7. Close Connection

In [ ]:
# Close semua koneksi database
try:
    cursor_old.close()
    cursor_new.close()
    db_old.close()
    db_new.close()
    print("✓ Database connections closed")
except:
    print("⚠ Error closing connections (mungkin sudah tertutup)")